ARIMA PIPELINE — Dự đoán people_vaccinated_per_hundred

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error


THAM SỐ TOÀN CỤC

In [2]:
ROLLING_WINDOW        = 14
Z_THRESHOLD           = 3.0
OUTLIER_PCT_THRESHOLD = 1.0
TARGET_COL            = "people_vaccinated_per_hundred"
TEST_WEEKS            = 52
P_RANGE               = range(0, 4)
Q_RANGE               = range(0, 4)
WF_MIN_TRAIN_WEEKS    = 52
OUTPUT_PATH           = "data_ready.csv"

countries = [
    'Vietnam', 'United States', 'China',
    'United Kingdom', 'Brazil', 'India', 'South Africa'
]



HÀM TIỆN ÍCH

In [3]:
def safe_cols(col_list, dataframe):
    """Chỉ trả về các cột thực sự tồn tại trong dataframe."""
    return [c for c in col_list if c in dataframe.columns]

def mape(actual, predicted):
    """Mean Absolute Percentage Error — bỏ qua các điểm actual == 0."""
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    mask = actual != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

def rolling_zscore_mask(series, window, threshold):
    """Trả về boolean mask: True = outlier theo rolling z-score."""
    roll_mean = series.rolling(window, center=True, min_periods=3).mean()
    roll_std  = series.rolling(window, center=True, min_periods=3).std()
    z = (series - roll_mean) / roll_std.replace(0, np.nan)
    return z.abs() > threshold

def replace_outliers_rolling_median(group, col, window, threshold):
    """Thay outlier bằng rolling median; fallback median toàn chuỗi; clip >= 0."""
    series   = group[col].copy()
    roll_med = series.rolling(window, center=True, min_periods=3).median()
    roll_std = series.rolling(window, center=True, min_periods=3).std()
    z        = (series - roll_med) / roll_std.replace(0, np.nan)
    outlier_mask = z.abs() > threshold
    series[outlier_mask] = roll_med[outlier_mask]
    series = series.fillna(series.median())
    series = series.clip(lower=0)
    return series

def test_stationarity(series, name, country):
    """Kiểm tra stationarity bằng ADF + KPSS. Trả về dict kết quả."""
    series = series.dropna()
    if len(series) < 20:
        return {
            "country": country, "series": name,
            "adf_p": None, "kpss_p": None,
            "adf_stat": "N/A", "kpss_stat": "N/A",
            "conclusion": "⚠️  Quá ít dữ liệu"
        }

    adf_result  = adfuller(series, autolag="AIC")
    adf_pval    = adf_result[1]
    kpss_result = kpss(series, regression="c", nlags="auto")
    kpss_pval   = kpss_result[1]

    adf_stat  = "Stat ✅"    if adf_pval < 0.05  else "Non-stat ❌"
    kpss_stat = "Stat ✅"    if kpss_pval > 0.05 else "Non-stat ❌"

    if adf_pval < 0.05 and kpss_pval > 0.05:
        conclusion = "✅ STATIONARY"
    elif adf_pval >= 0.05 and kpss_pval <= 0.05:
        conclusion = "❌ NON-STATIONARY"
    else:
        conclusion = "⚠️  KẾT QUẢ MÂU THUẪN — cần xem xét thêm"

    return {
        "country": country, "series": name,
        "adf_p":  round(adf_pval, 4),  "adf_stat":  adf_stat,
        "kpss_p": round(kpss_pval, 4), "kpss_stat": kpss_stat,
        "conclusion": conclusion
    }



1) ĐỌC DỮ LIỆU

In [4]:
print("\n" + "=" * 60)
print("BƯỚC 1 — Đọc dữ liệu")
print("=" * 60)

df = pd.read_csv('covid_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country', 'date']).reset_index(drop=True)

rows, n_cols = df.shape
print(f"  Số hàng: {rows:,}")
print(f"  Số cột : {n_cols}")


BƯỚC 1 — Đọc dữ liệu
  Số hàng: 570,606
  Số cột : 61


2. DROP CÁC CỘT KHÔNG CẦN THIẾT VÀ LỌC QUỐC GIA

In [5]:
print("\n" + "=" * 60)
print("BƯỚC 2 — Drop cột không cần & lọc quốc gia")
print("=" * 60)

cols_to_drop = [
    'new_tests', 'new_tests_per_thousand', 'total_tests', 'total_tests_per_thousand',
    'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
    'positive_rate', 'tests_per_case',
    'total_vaccinations', 'total_vaccinations_per_hundred',
    'new_vaccinations', 'new_vaccinations_smoothed', 'human_development_index',
    'new_vaccinations_smoothed_per_million', 'new_people_vaccinated_smoothed',
    'new_people_vaccinated_smoothed_per_hundred',
    'reproduction_rate', 'stringency_index', 'hosp_patients_per_million',
    'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million',
    'icu_patients', 'icu_patients_per_million',
    'weekly_icu_admissions', 'weekly_icu_admissions_per_million',
    'total_boosters', 'total_boosters_per_hundred',
    'handwashing_facilities', 'excess_mortality',
    'excess_mortality_cumulative', 'excess_mortality_cumulative_absolute',
    'excess_mortality_cumulative_per_million', 'hosp_patients'
]
# Chỉ drop cột tồn tại — tránh KeyError
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# Lọc quốc gia TRƯỚC khi xử lý để tiết kiệm tài nguyên
df = df[df['country'].isin(countries)].reset_index(drop=True)

rows, n_cols = df.shape
print(f"  Sau drop & lọc: {rows:,} hàng × {n_cols} cột")
print(f"\n  Missing values:\n{df.isnull().sum().to_string()}")


BƯỚC 2 — Drop cột không cần & lọc quốc gia
  Sau drop & lọc: 15,715 hàng × 28 cột

  Missing values:
country                                    0
date                                       0
total_cases                               21
new_cases                               1033
new_cases_smoothed                      1064
total_cases_per_million                   21
new_cases_per_million                   1033
new_cases_smoothed_per_million          1064
total_deaths                              21
new_deaths                                24
new_deaths_smoothed                       56
total_deaths_per_million                  21
new_deaths_per_million                    24
new_deaths_smoothed_per_million           56
people_vaccinated                      11325
people_fully_vaccinated                11489
people_vaccinated_per_hundred          11325
people_fully_vaccinated_per_hundred    11489
code                                       0
continent                                  

3. XỬ LÝ GIÁ TRỊ THIẾU

In [6]:
print("\n" + "=" * 60)
print("BƯỚC 3 — Xử lý Missing Values")
print("=" * 60)

STATIC_COLS = safe_cols([
    "code", "continent", "population", "population_density",
    "median_age", "life_expectancy", "gdp_per_capita",
    "extreme_poverty", "diabetes_prevalence", "hospital_beds_per_thousand",
], df)

TS_CUMULATIVE_COLS = safe_cols([
    "total_cases", "total_cases_per_million",
    "total_deaths", "total_deaths_per_million",
    "people_vaccinated", "people_vaccinated_per_hundred",
    "people_fully_vaccinated", "people_fully_vaccinated_per_hundred",
], df)

TS_DAILY_COLS = safe_cols([
    "new_cases", "new_cases_per_million",
    "new_cases_smoothed", "new_cases_smoothed_per_million",
    "new_deaths", "new_deaths_per_million",
    "new_deaths_smoothed", "new_deaths_smoothed_per_million",
], df)

# 3a. Static cols: ffill + bfill, fallback median (số) hoặc mode (chuỗi)
missing_before = df[STATIC_COLS].isnull().sum().sum()
df[STATIC_COLS] = df.groupby("country")[STATIC_COLS].transform(
    lambda x: x.ffill().bfill()
)
for col in STATIC_COLS:
    if df[col].dtype in [np.float64, np.int64, float, int]:
        df[col] = df[col].fillna(df[col].median())
    else:
        mode_val = df[col].mode()
        if len(mode_val) > 0:
            df[col] = df[col].fillna(mode_val[0])
print(f"  Static cols     — đã fill {missing_before - df[STATIC_COLS].isnull().sum().sum():,} ô")

# 3b. Cumulative ts: ffill rồi fill 0
missing_before = df[TS_CUMULATIVE_COLS].isnull().sum().sum()
df[TS_CUMULATIVE_COLS] = df.groupby("country")[TS_CUMULATIVE_COLS].transform(
    lambda x: x.ffill().fillna(0)
)
print(f"  Cumulative ts   — đã fill {missing_before - df[TS_CUMULATIVE_COLS].isnull().sum().sum():,} ô")

# 3c. Daily ts: ffill rồi fill 0
missing_before = df[TS_DAILY_COLS].isnull().sum().sum()
df[TS_DAILY_COLS] = df.groupby("country")[TS_DAILY_COLS].transform(
    lambda x: x.ffill().fillna(0)
)
print(f"  Daily ts        — đã fill {missing_before - df[TS_DAILY_COLS].isnull().sum().sum():,} ô")

remaining_missing = df.isnull().sum().sum()
print(f"\n  ✅ Missing còn lại: {remaining_missing}")


BƯỚC 3 — Xử lý Missing Values
  Static cols     — đã fill 2,245 ô
  Cumulative ts   — đã fill 45,712 ô
  Daily ts        — đã fill 4,354 ô

  ✅ Missing còn lại: 0


4. PHÁT HIỆN OUTLIERS (Rolling Z-score)

In [7]:
print("\n" + "=" * 60)
print("BƯỚC 4 — Quét Outliers")
print(f"         window={ROLLING_WINDOW} | z={Z_THRESHOLD} | xử lý nếu >{OUTLIER_PCT_THRESHOLD}%")
print("=" * 60)

NON_OUTLIER_COLS = {"population"}
numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in NON_OUTLIER_COLS
]
print(f"\n  Tổng cột số cần quét: {len(numeric_cols)}")

scan_results   = {}
COLS_TO_HANDLE = []

print(f"\n  {'Cột':<45} {'Outliers':>10} {'%':>7}  Quyết định")
print("  " + "-" * 78)

for col in numeric_cols:
    mask = df.groupby("country")[col].transform(
        lambda x: rolling_zscore_mask(x, ROLLING_WINDOW, Z_THRESHOLD)
    ).fillna(False)

    cnt = int(mask.sum())
    pct = cnt / len(df) * 100
    scan_results[col] = {"count": cnt, "pct": pct}

    if cnt == 0:
        decision = "✅ Không có"
    elif pct < OUTLIER_PCT_THRESHOLD:
        decision = f"⚠️  Ít ({cnt} điểm) — bỏ qua"
    else:
        decision = "🔴 Xử lý"
        COLS_TO_HANDLE.append(col)

    print(f"  {col:<45} {cnt:>10,} {pct:>6.2f}%  {decision}")

print(f"\n  → Tổng cột sẽ xử lý: {len(COLS_TO_HANDLE)}")


BƯỚC 4 — Quét Outliers
         window=14 | z=3.0 | xử lý nếu >1.0%

  Tổng cột số cần quét: 23

  Cột                                             Outliers       %  Quyết định
  ------------------------------------------------------------------------------
  total_cases                                            0   0.00%  ✅ Không có
  new_cases                                             77   0.49%  ⚠️  Ít (77 điểm) — bỏ qua
  new_cases_smoothed                                     3   0.02%  ⚠️  Ít (3 điểm) — bỏ qua
  total_cases_per_million                                0   0.00%  ✅ Không có
  new_cases_per_million                                 77   0.49%  ⚠️  Ít (77 điểm) — bỏ qua
  new_cases_smoothed_per_million                         3   0.02%  ⚠️  Ít (3 điểm) — bỏ qua
  total_deaths                                           0   0.00%  ✅ Không có
  new_deaths                                            88   0.56%  ⚠️  Ít (88 điểm) — bỏ qua
  new_deaths_smoothed                

5. XỬ LÝ OUTLIERS (Rolling Median Replacement)

In [8]:
print("\n" + "=" * 60)
print("BƯỚC 5 — Xử lý Outliers (Rolling Median Replacement)")
print("=" * 60)

for col in COLS_TO_HANDLE:
    df[col] = df.groupby("country", group_keys=False).apply(
        lambda g: replace_outliers_rolling_median(g, col, ROLLING_WINDOW, Z_THRESHOLD)
    )
    print(f"  ✅ {col:<45} ({scan_results[col]['count']:,} điểm thay thế)")

# Kiểm tra lại
still_outliers = []
for col in COLS_TO_HANDLE:
    mask = df.groupby("country")[col].transform(
        lambda x: rolling_zscore_mask(x, ROLLING_WINDOW, Z_THRESHOLD)
    ).fillna(False)
    cnt = int(mask.sum())
    if cnt > 0:
        still_outliers.append((col, cnt))

if not still_outliers:
    print("\n  ✅ Tất cả outliers đã được xử lý!")
else:
    print(f"\n  ⚠️  Còn {len(still_outliers)} cột vẫn có outlier cực đoan:")
    for col, cnt in still_outliers:
        print(f"     {col:<45} {cnt:>6,} điểm")


BƯỚC 5 — Xử lý Outliers (Rolling Median Replacement)

  ✅ Tất cả outliers đã được xử lý!


6. WEEKLY RESAMPLING

In [9]:
print("\n" + "=" * 60)
print("BƯỚC 6 — Weekly Resampling (W-MON)")
print("=" * 60)

RESAMPLE_FIRST = safe_cols([
    "code", "continent", "population", "population_density",
    "median_age", "life_expectancy", "gdp_per_capita",
    "extreme_poverty", "diabetes_prevalence", "hospital_beds_per_thousand",
], df)

RESAMPLE_LAST = safe_cols([
    "total_cases", "total_cases_per_million",
    "total_deaths", "total_deaths_per_million",
    "people_vaccinated", "people_vaccinated_per_hundred",
    "people_fully_vaccinated", "people_fully_vaccinated_per_hundred",
], df)

already_assigned = set(RESAMPLE_FIRST + RESAMPLE_LAST + ["country", "date"])
RESAMPLE_MEAN = [
    c for c in df.columns
    if c not in already_assigned
    and df[c].dtype in [np.float64, np.int64, float, int]
]

print(f"\n  first (static)   : {len(RESAMPLE_FIRST)} cột")
print(f"  last  (luỹ kế)   : {len(RESAMPLE_LAST)} cột")
print(f"  mean  (hàng ngày): {len(RESAMPLE_MEAN)} cột")

def resample_country(group):
    group = group.set_index("date")
    agg_first = group[RESAMPLE_FIRST].resample("W-MON").first() if RESAMPLE_FIRST else pd.DataFrame()
    agg_last  = group[RESAMPLE_LAST].resample("W-MON").last()  if RESAMPLE_LAST  else pd.DataFrame()
    agg_mean  = group[RESAMPLE_MEAN].resample("W-MON").mean()  if RESAMPLE_MEAN  else pd.DataFrame()
    combined  = pd.concat([agg_first, agg_last, agg_mean], axis=1)
    combined["country"] = group["country"].iloc[0]
    return combined.reset_index().rename(columns={"date": "week_start"})

print("\n  Đang resample...")
weekly_parts = []
for country in countries:
    grp = df[df["country"] == country].copy()
    weekly_parts.append(resample_country(grp))

df_weekly = pd.concat(weekly_parts, ignore_index=True)

cols_order = ["country", "week_start"] + [
    c for c in df_weekly.columns if c not in ("country", "week_start")
]
df_weekly = df_weekly[cols_order]

print(f"\n  Daily  : {df.shape[0]:,} hàng × {df.shape[1]} cột")
print(f"  Weekly : {df_weekly.shape[0]:,} hàng × {df_weekly.shape[1]} cột")
print(f"  Range  : {df_weekly['week_start'].min().date()} → {df_weekly['week_start'].max().date()}")

print("\n  Số tuần mỗi quốc gia:")
for country in countries:
    n = len(df_weekly[df_weekly["country"] == country])
    print(f"     {country:<20} {n:>4} tuần")

# Fill missing phát sinh sau resample
missing_after = df_weekly.isnull().sum()
missing_after = missing_after[missing_after > 0]
if missing_after.empty:
    print("\n  ✅ Không có missing sau resample!")
else:
    print(f"\n  ⚠️  {len(missing_after)} cột missing sau resample — forward-fill lại:")
    df_weekly[missing_after.index.tolist()] = (
        df_weekly.groupby("country")[missing_after.index.tolist()]
        .transform(lambda x: x.ffill().bfill())
    )
    print("  ✅ Đã fill xong!")


BƯỚC 6 — Weekly Resampling (W-MON)

  first (static)   : 10 cột
  last  (luỹ kế)   : 8 cột
  mean  (hàng ngày): 8 cột

  Đang resample...

  Daily  : 15,715 hàng × 28 cột
  Weekly : 2,247 hàng × 28 cột
  Range  : 2020-01-06 → 2026-02-23

  Số tuần mỗi quốc gia:
     Vietnam               321 tuần
     United States         321 tuần
     China                 321 tuần
     United Kingdom        321 tuần
     Brazil                321 tuần
     India                 321 tuần
     South Africa          321 tuần

  ✅ Không có missing sau resample!


7. DATA TRANSFORMATION (Differencing)

In [10]:
print("\n" + "=" * 60)
print("BƯỚC 7 — Data Transformation (First-order Differencing)")
print("=" * 60)
print(f"""
  Target : '{TARGET_COL}'
  Loại   : luỹ kế, giới hạn [0, 100]
  Lý do  : Log transform không phù hợp vì chuỗi bị chặn tại 100
  Cách   : diff(1) → tốc độ tiêm chủng mỗi tuần
""")

DIFF_COL  = f"{TARGET_COL}_diff1"
DIFF2_COL = f"{TARGET_COL}_diff2"

for country in countries:
    mask = df_weekly["country"] == country
    df_weekly.loc[mask, DIFF_COL] = df_weekly.loc[mask, TARGET_COL].diff(1)

df_weekly[DIFF_COL] = df_weekly.groupby("country")[DIFF_COL].transform(
    lambda x: x.fillna(0)
)

print(f"  {'Quốc gia':<20} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")
print("  " + "-" * 56)
for country in countries:
    sub = df_weekly[df_weekly["country"] == country][DIFF_COL]
    print(f"  {country:<20} {sub.min():>8.3f} {sub.max():>8.3f} {sub.mean():>8.3f} {sub.std():>8.3f}")




BƯỚC 7 — Data Transformation (First-order Differencing)

  Target : 'people_vaccinated_per_hundred'
  Loại   : luỹ kế, giới hạn [0, 100]
  Lý do  : Log transform không phù hợp vì chuỗi bị chặn tại 100
  Cách   : diff(1) → tốc độ tiêm chủng mỗi tuần

  Quốc gia                  Min      Max     Mean      Std
  --------------------------------------------------------
  Vietnam                 0.000    7.429    0.283    0.953
  United States           0.000    4.109    0.246    0.636
  China                   0.000   43.644    0.288    3.043
  United Kingdom          0.000    5.116    0.246    0.778
  Brazil                  0.000    4.167    0.281    0.731
  India                   0.000    2.959    0.225    0.546
  South Africa            0.000    2.366    0.121    0.321


8. KIỂM TRA STATIONARITY (ADF + KPSS)

In [11]:
print("\n" + "=" * 60)
print("BƯỚC 8 — Kiểm tra Stationarity")
print("=" * 60)
print("""
  ADF  : H0 = có unit root → p < 0.05 → Stationary ✅
  KPSS : H0 = stationary   → p > 0.05 → Stationary ✅
  Kết luận chắc chắn khi CẢ HAI đồng thuận.
""")

stat_results = []

for country in countries:
    sub = df_weekly[df_weekly["country"] == country]
    stat_results.append(test_stationarity(sub[TARGET_COL], "original", country))
    stat_results.append(test_stationarity(sub[DIFF_COL],   "diff(1)",  country))

print(f"  {'Quốc gia':<20} {'Chuỗi':<12} {'ADF p':>8} {'ADF':>12} {'KPSS p':>8} {'KPSS':>12}  Kết luận")
print("  " + "-" * 100)
for r in stat_results:
    adf_p_str  = f"{r['adf_p']:.4f}"  if r['adf_p']  is not None else "  N/A  "
    kpss_p_str = f"{r['kpss_p']:.4f}" if r['kpss_p'] is not None else "  N/A  "
    print(
        f"  {r['country']:<20} {r['series']:<12} "
        f"{adf_p_str:>8} {r['adf_stat']:>12} "
        f"{kpss_p_str:>8} {r['kpss_stat']:>12}  {r['conclusion']}"
    )

# Tự động thêm diff(2) nếu cần
needs_diff2 = [
    r["country"] for r in stat_results
    if r["series"] == "diff(1)" and "NON-STATIONARY" in r["conclusion"]
]

if not needs_diff2:
    print("\n  ✅ Tất cả stationary sau diff(1)")
else:
    print(f"\n  ⚠️  {len(needs_diff2)} quốc gia cần diff(2): {needs_diff2}")
    for country in needs_diff2:
        mask = df_weekly["country"] == country
        df_weekly.loc[mask, DIFF2_COL] = df_weekly.loc[mask, DIFF_COL].diff(1)
    df_weekly[DIFF2_COL] = df_weekly.groupby("country")[DIFF2_COL].transform(
        lambda x: x.fillna(0)
    )
    print(f"\n  Kết quả sau diff(2):")
    print(f"  {'Quốc gia':<20} {'ADF p':>8} {'KPSS p':>8}  Kết luận")
    print("  " + "-" * 60)
    for country in needs_diff2:
        sub = df_weekly[df_weekly["country"] == country]
        r2  = test_stationarity(sub[DIFF2_COL], "diff(2)", country)
        print(f"  {country:<20} {r2['adf_p']:>8.4f} {r2['kpss_p']:>8.4f}  {r2['conclusion']}")




BƯỚC 8 — Kiểm tra Stationarity

  ADF  : H0 = có unit root → p < 0.05 → Stationary ✅
  KPSS : H0 = stationary   → p > 0.05 → Stationary ✅
  Kết luận chắc chắn khi CẢ HAI đồng thuận.

  Quốc gia             Chuỗi           ADF p          ADF   KPSS p         KPSS  Kết luận
  ----------------------------------------------------------------------------------------------------
  Vietnam              original       0.3786   Non-stat ❌   0.0100   Non-stat ❌  ❌ NON-STATIONARY
  Vietnam              diff(1)        0.0621   Non-stat ❌   0.0998       Stat ✅  ⚠️  KẾT QUẢ MÂU THUẪN — cần xem xét thêm
  United States        original       0.1183   Non-stat ❌   0.0100   Non-stat ❌  ❌ NON-STATIONARY
  United States        diff(1)        0.0992   Non-stat ❌   0.0232   Non-stat ❌  ❌ NON-STATIONARY
  China                original       0.2854   Non-stat ❌   0.0100   Non-stat ❌  ❌ NON-STATIONARY
  China                diff(1)        0.0535   Non-stat ❌   0.0992       Stat ✅  ⚠️  KẾT QUẢ MÂU THUẪN — cần 

/tmp/ipykernel_1549/3025596080.py:46: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_result = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_1549/3025596080.py:46: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_result = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_1549/3025596080.py:46: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_result = kpss(series, regression="c", nlags="auto")
/tmp/ipykernel_1549/3025596080.py:46: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  k

9. PHÂN RÃ CHUỖI THỜI GIAN (Seasonal Decompose)

In [12]:
print("\n" + "=" * 60)
print("BƯỚC 9 — Phân rã chuỗi thời gian (Seasonal Decompose)")
print("=" * 60)
print("  Model: additive | Period: 52 tuần\n")

decomp_summary = {}

for country in countries:
    sub = df_weekly[df_weekly["country"] == country].copy()
    sub = sub.set_index("week_start")[TARGET_COL].dropna()

    if len(sub) < 104:
        print(f"  ⚠️  {country:<20} — quá ít dữ liệu ({len(sub)} tuần), bỏ qua")
        decomp_summary[country] = {"trend_strength": 0, "seasonal_strength": 0, "residual_std": 0}
        continue

    try:
        decomp = seasonal_decompose(sub, model="additive", period=52, extrapolate_trend="freq")

        seasonal_strength = np.std(decomp.seasonal) / (
            np.std(decomp.seasonal) + np.std(decomp.resid.dropna())
        )
        trend_strength = 1 - np.var(decomp.resid.dropna()) / np.var(
            decomp.trend.dropna() + decomp.resid.dropna()
        )
        residual_std = np.std(decomp.resid.dropna())

        decomp_summary[country] = {
            "trend_strength":    round(trend_strength, 3),
            "seasonal_strength": round(seasonal_strength, 3),
            "residual_std":      round(residual_std, 3),
        }

        fig, axes = plt.subplots(4, 1, figsize=(14, 10))
        fig.suptitle(f"Decomposition — {country}", fontsize=13, fontweight="bold")

        axes[0].plot(sub.index, sub.values, color="#2563EB", linewidth=1.2)
        axes[0].set_ylabel("Original"); axes[0].grid(True, alpha=0.3)

        axes[1].plot(decomp.trend.index, decomp.trend.values, color="#16A34A", linewidth=1.5)
        axes[1].set_ylabel("Trend"); axes[1].grid(True, alpha=0.3)
        axes[1].annotate(f"Trend strength: {trend_strength:.2f}",
                         xy=(0.02, 0.85), xycoords="axes fraction",
                         fontsize=9, color="#16A34A", fontweight="bold")

        axes[2].plot(decomp.seasonal.index, decomp.seasonal.values, color="#D97706", linewidth=1.0)
        axes[2].set_ylabel("Seasonal"); axes[2].grid(True, alpha=0.3)
        axes[2].annotate(f"Seasonal strength: {seasonal_strength:.2f}",
                         xy=(0.02, 0.85), xycoords="axes fraction",
                         fontsize=9, color="#D97706", fontweight="bold")

        axes[3].plot(decomp.resid.index, decomp.resid.values, color="#DC2626", linewidth=0.8, alpha=0.8)
        axes[3].axhline(0, color="black", linewidth=0.8, linestyle="--")
        axes[3].set_ylabel("Residual"); axes[3].grid(True, alpha=0.3)

        plt.tight_layout()
        fname = f"decomposition_{country.replace(' ', '_')}.png"
        plt.savefig(fname, dpi=120, bbox_inches="tight")
        plt.close()

        strength_label = "mạnh" if seasonal_strength > 0.4 else "yếu"
        print(f"  {country:<20} trend={trend_strength:.2f} | seasonal={seasonal_strength:.2f} ({strength_label}) → 💾 {fname}")

    except Exception as e:
        print(f"  ⚠️  {country:<20} — lỗi: {e}")
        decomp_summary[country] = {"trend_strength": 0, "seasonal_strength": 0, "residual_std": 0}




BƯỚC 9 — Phân rã chuỗi thời gian (Seasonal Decompose)
  Model: additive | Period: 52 tuần

  Vietnam              trend=0.98 | seasonal=0.30 (yếu) → 💾 decomposition_Vietnam.png
  United States        trend=0.96 | seasonal=0.24 (yếu) → 💾 decomposition_United_States.png
  China                trend=0.96 | seasonal=0.20 (yếu) → 💾 decomposition_China.png
  United Kingdom       trend=0.94 | seasonal=0.25 (yếu) → 💾 decomposition_United_Kingdom.png
  Brazil               trend=0.97 | seasonal=0.12 (yếu) → 💾 decomposition_Brazil.png
  India                trend=0.98 | seasonal=0.19 (yếu) → 💾 decomposition_India.png
  South Africa         trend=0.99 | seasonal=0.24 (yếu) → 💾 decomposition_South_Africa.png


10. ACF / PACF PLOT

In [13]:
print("\n" + "=" * 60)
print("BƯỚC 10 — ACF / PACF (xác định p, q)")
print("=" * 60)

acf_pacf_summary = {}

for country in countries:
    sub    = df_weekly[df_weekly["country"] == country].copy()
    series = sub[DIFF_COL].dropna()

    if len(series) < 30:
        print(f"  ⚠️  {country:<20} — quá ít dữ liệu, bỏ qua")
        acf_pacf_summary[country] = {"suggest_p": 1, "suggest_q": 1}
        continue

    lags = min(40, len(series) // 2 - 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"ACF & PACF — {country}", fontsize=12, fontweight="bold")

    plot_acf( series, ax=axes[0], lags=lags, alpha=0.05, color="#2563EB")
    plot_pacf(series, ax=axes[1], lags=lags, alpha=0.05, color="#DC2626", method="ywm")

    for ax in axes:
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("Lag (tuần)")

    acf_vals,  acf_confint  = acf( series, nlags=lags, alpha=0.05)
    pacf_vals, pacf_confint = pacf(series, nlags=lags, alpha=0.05, method="ywm")

    sig_acf  = [i for i in range(1, len(acf_vals))
                if acf_vals[i] > acf_confint[i, 1] - acf_vals[i]
                or acf_vals[i] < acf_confint[i, 0] - acf_vals[i]]
    sig_pacf = [i for i in range(1, len(pacf_vals))
                if pacf_vals[i] > pacf_confint[i, 1] - pacf_vals[i]
                or pacf_vals[i] < pacf_confint[i, 0] - pacf_vals[i]]

    suggest_q = sig_acf[0]  if sig_acf  else 1
    suggest_p = sig_pacf[0] if sig_pacf else 1

    acf_pacf_summary[country] = {"suggest_p": suggest_p, "suggest_q": suggest_q}

    axes[0].set_title(f"ACF  → gợi ý q = {suggest_q}", fontsize=10)
    axes[1].set_title(f"PACF → gợi ý p = {suggest_p}", fontsize=10)

    plt.tight_layout()
    fname = f"acf_pacf_{country.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.close()

    print(f"  {country:<20} gợi ý p={suggest_p}, q={suggest_q}  → 💾 {fname}")




BƯỚC 10 — ACF / PACF (xác định p, q)
  Vietnam              gợi ý p=1, q=1  → 💾 acf_pacf_Vietnam.png
  United States        gợi ý p=1, q=1  → 💾 acf_pacf_United_States.png
  China                gợi ý p=11, q=11  → 💾 acf_pacf_China.png
  United Kingdom       gợi ý p=1, q=1  → 💾 acf_pacf_United_Kingdom.png
  Brazil               gợi ý p=1, q=1  → 💾 acf_pacf_Brazil.png
  India                gợi ý p=1, q=1  → 💾 acf_pacf_India.png
  South Africa         gợi ý p=1, q=1  → 💾 acf_pacf_South_Africa.png


11. XÁC ĐỊNH d VÀ CHUỖI INPUT ARIMA

In [14]:
print("\n" + "=" * 60)
print("BƯỚC 11 — Xác định d và chuỗi input ARIMA")
print("=" * 60)

# Thêm datetime features (metadata / EDA, KHÔNG đưa vào ARIMA)
df_weekly["year"]         = df_weekly["week_start"].dt.year
df_weekly["month"]        = df_weekly["week_start"].dt.month
df_weekly["quarter"]      = df_weekly["week_start"].dt.quarter
df_weekly["week_of_year"] = df_weekly["week_start"].dt.isocalendar().week.astype(int)
df_weekly["is_q1"]        = (df_weekly["quarter"] == 1).astype(int)
df_weekly["is_q4"]        = (df_weekly["quarter"] == 4).astype(int)
print("  ⚠️  DateTime features (year/month/quarter/...) chỉ dùng EDA, KHÔNG đưa vào ARIMA")

arima_input_col = {}
print(f"\n  {'Quốc gia':<20} {'d':>3}  Chuỗi input ARIMA")
print("  " + "-" * 55)

for country in countries:
    non_stat_orig = any(
        r["conclusion"] == "❌ NON-STATIONARY"
        for r in stat_results if r["country"] == country and r["series"] == "original"
    )
    stat_d1 = any(
        r["conclusion"] == "✅ STATIONARY"
        for r in stat_results if r["country"] == country and r["series"] == "diff(1)"
    )

    if not non_stat_orig:
        d, col = 0, TARGET_COL
    elif stat_d1:
        d, col = 1, DIFF_COL
    else:
        d, col = 2, DIFF2_COL if DIFF2_COL in df_weekly.columns else DIFF_COL

    arima_input_col[country] = {"d": d, "col": col}
    print(f"  {country:<20} {d:>3}  '{col}'")




BƯỚC 11 — Xác định d và chuỗi input ARIMA
  ⚠️  DateTime features (year/month/quarter/...) chỉ dùng EDA, KHÔNG đưa vào ARIMA

  Quốc gia               d  Chuỗi input ARIMA
  -------------------------------------------------------
  Vietnam                2  'people_vaccinated_per_hundred_diff2'
  United States          2  'people_vaccinated_per_hundred_diff2'
  China                  2  'people_vaccinated_per_hundred_diff2'
  United Kingdom         2  'people_vaccinated_per_hundred_diff2'
  Brazil                 2  'people_vaccinated_per_hundred_diff2'
  India                  2  'people_vaccinated_per_hundred_diff2'
  South Africa           2  'people_vaccinated_per_hundred_diff2'


12. TRAIN / TEST SPLIT (theo thời gian)

In [15]:
print("\n" + "=" * 60)
print(f"BƯỚC 12 — Train/Test Split (test = {TEST_WEEKS} tuần cuối)")
print("=" * 60)
print("  ⚠️  Time-based split — KHÔNG random (tránh data leakage)\n")

split_summary = {}

for country in countries:
    sub       = df_weekly[df_weekly["country"] == country].copy().reset_index(drop=True)
    input_col = arima_input_col[country]["col"]
    series    = sub[input_col].dropna()

    if len(series) <= TEST_WEEKS:
        print(f"  ⚠️  {country:<20} — quá ít dữ liệu")
        continue

    n_train = len(series) - TEST_WEEKS
    split_summary[country] = {
        "train_start": sub["week_start"].iloc[0].date(),
        "train_end":   sub["week_start"].iloc[n_train - 1].date(),
        "test_start":  sub["week_start"].iloc[n_train].date(),
        "test_end":    sub["week_start"].iloc[-1].date(),
        "n_train":     n_train,
        "n_test":      TEST_WEEKS,
    }
    print(
        f"  {country:<20} "
        f"Train: {n_train:>4} tuần ({sub['week_start'].iloc[0].date()} → {sub['week_start'].iloc[n_train-1].date()})  |  "
        f"Test: {TEST_WEEKS:>4} tuần ({sub['week_start'].iloc[n_train].date()} → {sub['week_start'].iloc[-1].date()})"
    )

df_weekly["split"] = "train"
for country, info in split_summary.items():
    mask_test = (
        (df_weekly["country"] == country) &
        (df_weekly["week_start"] >= pd.Timestamp(info["test_start"]))
    )
    df_weekly.loc[mask_test, "split"] = "test"

print(f"\n  Tổng Train : {(df_weekly['split']=='train').sum():,}")
print(f"  Tổng Test  : {(df_weekly['split']=='test').sum():,}")




BƯỚC 12 — Train/Test Split (test = 52 tuần cuối)
  ⚠️  Time-based split — KHÔNG random (tránh data leakage)

  Vietnam              Train:  269 tuần (2020-01-06 → 2025-02-24)  |  Test:   52 tuần (2025-03-03 → 2026-02-23)
  United States        Train:  269 tuần (2020-01-06 → 2025-02-24)  |  Test:   52 tuần (2025-03-03 → 2026-02-23)
  China                Train:  269 tuần (2020-01-06 → 2025-02-24)  |  Test:   52 tuần (2025-03-03 → 2026-02-23)
  United Kingdom       Train:  269 tuần (2020-01-06 → 2025-02-24)  |  Test:   52 tuần (2025-03-03 → 2026-02-23)
  Brazil               Train:  269 tuần (2020-01-06 → 2025-02-24)  |  Test:   52 tuần (2025-03-03 → 2026-02-23)
  India                Train:  269 tuần (2020-01-06 → 2025-02-24)  |  Test:   52 tuần (2025-03-03 → 2026-02-23)
  South Africa         Train:  269 tuần (2020-01-06 → 2025-02-24)  |  Test:   52 tuần (2025-03-03 → 2026-02-23)

  Tổng Train : 1,883
  Tổng Test  : 364


13. GRID SEARCH (p, d, q) — tiêu chí AIC

In [16]:
print("\n" + "=" * 65)
print("BƯỚC 13 — Grid Search tham số ARIMA")
print(f"          p ∈ {list(P_RANGE)}, q ∈ {list(Q_RANGE)} | tiêu chí: AIC")
print("=" * 65)

best_params = {}

for country in countries:
    sub     = df_weekly[df_weekly["country"] == country]
    col     = arima_input_col[country]["col"]
    d       = arima_input_col[country]["d"]
    train_s = sub[sub["split"] == "train"][col].dropna().values

    if len(train_s) < WF_MIN_TRAIN_WEEKS:
        print(f"  ⚠️  {country:<20} — ít dữ liệu, dùng p=1, q=1")
        best_params[country] = {"p": 1, "d": d, "q": 1, "aic": None}
        continue

    best_aic     = np.inf
    best_pq      = (1, 1)
    results_grid = []

    for p, q in itertools.product(P_RANGE, Q_RANGE):
        if p == 0 and q == 0:
            continue
        try:
            fit = ARIMA(train_s, order=(p, d, q)).fit()
            results_grid.append((p, q, round(fit.aic, 2), round(fit.bic, 2)))
            if fit.aic < best_aic:
                best_aic = fit.aic
                best_pq  = (p, q)
        except Exception:
            continue

    best_params[country] = {
        "p": best_pq[0], "d": d, "q": best_pq[1],
        "aic": round(best_aic, 2)
    }

    results_grid.sort(key=lambda x: x[2])
    print(f"\n  {country} — Top 5 AIC:")
    print(f"    {'(p,d,q)':<12} {'AIC':>10} {'BIC':>10}")
    for row in results_grid[:5]:
        star = " ← best" if (row[0], row[1]) == best_pq else ""
        print(f"    ({row[0]},{d},{row[1]}){'':<6} {row[2]:>10.2f} {row[3]:>10.2f}{star}")




BƯỚC 13 — Grid Search tham số ARIMA
          p ∈ [0, 1, 2, 3], q ∈ [0, 1, 2, 3] | tiêu chí: AIC


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op


  Vietnam — Top 5 AIC:
    (p,d,q)             AIC        BIC
    (0,2,1)         -6069.42   -6062.24 ← best
    (1,2,0)         -6069.42   -6062.24
    (0,2,2)         -6067.42   -6056.66
    (1,2,1)         -6067.42   -6056.66
    (2,2,0)         -6067.42   -6056.66


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



  United States — Top 5 AIC:
    (p,d,q)             AIC        BIC
    (1,2,2)          -225.18    -210.83 ← best
    (2,2,2)          -223.69    -205.75
    (0,2,3)          -223.47    -209.12
    (0,2,2)          -212.19    -201.42
    (1,2,3)          -208.18    -190.24


...

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



  India — Top 5 AIC:
    (p,d,q)             AIC        BIC
    (2,2,3)           -42.81     -21.29 ← best
    (3,2,3)           -40.49     -15.38
    (0,2,3)           -39.64     -25.29
    (1,2,2)           -39.20     -24.85
    (1,2,3)           -38.03     -20.09


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



  South Africa — Top 5 AIC:
    (p,d,q)             AIC        BIC
    (0,2,3)           -86.89     -72.54 ← best
    (1,2,3)           -85.27     -67.33
    (3,2,2)           -83.41     -61.89
    (2,2,2)           -75.14     -57.20
    (3,2,3)           -70.46     -45.35


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


14. WALK-FORWARD VALIDATION

In [17]:
print("\n" + "=" * 65)
print("BƯỚC 14 — Walk-Forward Validation (Expanding Window)")
print("=" * 65)
print("""
  Tại mỗi bước t: train trên [0..t-1], predict bước t.
  Mô phỏng dự báo thực tế — không dùng dữ liệu tương lai.
""")

wf_results = {}

for country in countries:
    sub     = df_weekly[df_weekly["country"] == country].copy().reset_index(drop=True)
    col     = arima_input_col[country]["col"]
    p, d, q = best_params[country]["p"], best_params[country]["d"], best_params[country]["q"]
    series  = sub[col].dropna().values
    dates   = sub["week_start"].values

    if len(series) <= TEST_WEEKS + WF_MIN_TRAIN_WEEKS:
        print(f"  ⚠️  {country:<20} — không đủ dữ liệu walk-forward")
        continue

    n_train         = len(series) - TEST_WEEKS
    actuals, preds  = [], []

    for t in range(n_train, len(series)):
        train_window = series[:t]
        try:
            pred = ARIMA(train_window, order=(p, d, q)).fit().forecast(steps=1)[0]
        except Exception:
            pred = train_window[-1]   # fallback: naive
        actuals.append(series[t])
        preds.append(pred)

    mae_val  = mean_absolute_error(actuals, preds)
    rmse_val = np.sqrt(mean_squared_error(actuals, preds))
    mape_val = mape(actuals, preds)

    wf_results[country] = {
        "mae":     round(mae_val, 4),
        "rmse":    round(rmse_val, 4),
        "mape":    round(mape_val, 2),
        "actuals": actuals,
        "preds":   preds,
        "dates":   dates[n_train:]
    }
    print(f"  {country:<20} MAE={mae_val:.4f} | RMSE={rmse_val:.4f} | MAPE={mape_val:.2f}%")




BƯỚC 14 — Walk-Forward Validation (Expanding Window)

  Tại mỗi bước t: train trên [0..t-1], predict bước t.
  Mô phỏng dự báo thực tế — không dùng dữ liệu tương lai.



/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op

  Vietnam              MAE=0.0000 | RMSE=0.0000 | MAPE=nan%


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op

  United States        MAE=0.0024 | RMSE=0.0033 | MAPE=nan%


...

  Brazil               MAE=0.0088 | RMSE=0.0089 | MAPE=nan%


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op

  India                MAE=0.0119 | RMSE=0.0132 | MAPE=nan%


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op

  South Africa         MAE=0.0061 | RMSE=0.0064 | MAPE=nan%


15. TRAIN ARIMA CHÍNH THỨC & FORECAST

In [18]:
print("\n" + "=" * 65)
print("BƯỚC 15 — Train ARIMA chính thức & Forecast")
print("=" * 65)

final_results = {}
fig, axes = plt.subplots(len(countries), 1, figsize=(16, 5 * len(countries)))
if len(countries) == 1:
    axes = [axes]

for idx, country in enumerate(countries):
    sub      = df_weekly[df_weekly["country"] == country].copy().reset_index(drop=True)
    orig     = sub[TARGET_COL].values
    col      = arima_input_col[country]["col"]
    d        = arima_input_col[country]["d"]
    p, q     = best_params[country]["p"], best_params[country]["q"]
    series   = sub[col].dropna().values
    dates    = sub["week_start"].values
    n_train  = len(series) - TEST_WEEKS

    train_series = series[:n_train]
    test_series  = series[n_train:]

    try:
        model_fit    = ARIMA(train_series, order=(p, d, q)).fit()
        forecast_obj = model_fit.get_forecast(steps=TEST_WEEKS)
        forecast     = forecast_obj.predicted_mean
        conf_int     = forecast_obj.conf_int(alpha=0.05)

        mae_val  = mean_absolute_error(test_series, forecast)
        rmse_val = np.sqrt(mean_squared_error(test_series, forecast))
        mape_val = mape(test_series, forecast)

        final_results[country] = {
            "order": (p, d, q),
            "aic":   round(model_fit.aic, 2),
            "bic":   round(model_fit.bic, 2),
            "mae":   round(mae_val, 4),
            "rmse":  round(rmse_val, 4),
            "mape":  round(mape_val, 2),
        }

        print(f"\n  {country}")
        print(f"    ARIMA({p},{d},{q}) | AIC={model_fit.aic:.2f} | BIC={model_fit.bic:.2f}")
        print(f"    Test  — MAE={mae_val:.4f} | RMSE={rmse_val:.4f} | MAPE={mape_val:.2f}%")
        wf = wf_results.get(country, {})
        print(f"    WF    — MAE={wf.get('mae','N/A')} | RMSE={wf.get('rmse','N/A')} | MAPE={wf.get('mape','N/A')}%")
        print(model_fit.summary())

        # --- Inverse diff để plot trên chuỗi gốc ---
        last_val  = sub[TARGET_COL].values[n_train - 1]
        if d == 0:
            forecast_plot = forecast
        elif d == 1:
            forecast_plot = np.cumsum(forecast) + last_val
        else:
            last_diff     = sub[DIFF_COL].values[n_train - 1]
            forecast_plot = np.cumsum(np.cumsum(forecast)) + last_diff * TEST_WEEKS + last_val

        # --- Inverse diff CI ---
        if d == 0:
            ci_lower = conf_int.iloc[:, 0].values
            ci_upper = conf_int.iloc[:, 1].values
        elif d == 1:
            ci_lower = np.cumsum(conf_int.iloc[:, 0].values) + last_val
            ci_upper = np.cumsum(conf_int.iloc[:, 1].values) + last_val
        else:
            last_diff = sub[DIFF_COL].values[n_train - 1]
            ci_lower  = np.cumsum(np.cumsum(conf_int.iloc[:, 0].values)) + last_diff * TEST_WEEKS + last_val
            ci_upper  = np.cumsum(np.cumsum(conf_int.iloc[:, 1].values)) + last_diff * TEST_WEEKS + last_val

        # --- Plot ---
        ax = axes[idx]
        ax.plot(dates[:n_train], orig[:n_train],
                color="#2563EB", linewidth=1.2, label="Train (original)")
        ax.plot(dates[n_train:], orig[n_train:],
                color="#16A34A", linewidth=1.5, label="Actual (test)")
        ax.plot(dates[n_train:], forecast_plot,
                color="#DC2626", linewidth=1.5, linestyle="--",
                label=f"Forecast ARIMA({p},{d},{q})")
        ax.fill_between(dates[n_train:], ci_lower, ci_upper,
                        color="#DC2626", alpha=0.1, label="95% CI")
        ax.set_title(
            f"{country}  |  ARIMA({p},{d},{q})  "
            f"MAE={mae_val:.3f}  RMSE={rmse_val:.3f}  MAPE={mape_val:.1f}%",
            fontsize=11, fontweight="bold"
        )
        ax.set_ylabel(TARGET_COL, fontsize=9)
        ax.legend(fontsize=8, loc="upper left")
        ax.grid(True, alpha=0.3)

    except Exception as e:
        print(f"  ⚠️  {country} — lỗi fit: {e}")
        final_results[country] = {"order": (p, d, q), "error": str(e)}

plt.suptitle(f"ARIMA Forecast — '{TARGET_COL}'", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("arima_forecast_all_countries.png", dpi=130, bbox_inches="tight")
plt.close()
print("\n  📊 Đã lưu: arima_forecast_all_countries.png")


BƯỚC 15 — Train ARIMA chính thức & Forecast


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



  Vietnam
    ARIMA(0,2,1) | AIC=-6069.42 | BIC=-6062.24
    Test  — MAE=0.0000 | RMSE=0.0000 | MAPE=nan%
    WF    — MAE=0.0 | RMSE=0.0 | MAPE=nan%
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  269
Model:                 ARIMA(0, 2, 1)   Log Likelihood                3036.708
Date:                Wed, 29 Apr 2026   AIC                          -6069.417
Time:                        09:22:59   BIC                          -6062.242
Sample:                             0   HQIC                         -6066.535
                                - 269                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1               0   4.49e-19          0      1.000   -8.

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



  United States
    ARIMA(1,2,2) | AIC=-225.18 | BIC=-210.83
    Test  — MAE=0.0046 | RMSE=0.0049 | MAPE=nan%
    WF    — MAE=0.0024 | RMSE=0.0033 | MAPE=nan%
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  269
Model:                 ARIMA(1, 2, 2)   Log Likelihood                 116.588
Date:                Wed, 29 Apr 2026   AIC                           -225.176
Time:                        09:23:01   BIC                           -210.827
Sample:                             0   HQIC                          -219.412
                                - 269                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2634      0.030      8.645      0

16. TỔNG KẾT CUỐI

In [19]:
print("\n" + "=" * 65)
print("TỔNG KẾT KẾT QUẢ ARIMA")
print("=" * 65)
print(f"\n  {'Quốc gia':<20} {'Order':<14} {'AIC':>8} {'BIC':>8} {'MAE':>8} {'RMSE':>8} {'MAPE':>8}")
print("  " + "-" * 80)

for country in countries:
    r = final_results.get(country, {})
    if "error" in r:
        print(f"  {country:<20} ❌ Lỗi: {r['error']}")
        continue
    order_str = f"({r['order'][0]},{r['order'][1]},{r['order'][2]})"
    print(
        f"  {country:<20} {order_str:<14} {r['aic']:>8.2f} {r['bic']:>8.2f} "
        f"{r['mae']:>8.4f} {r['rmse']:>8.4f} {r['mape']:>7.2f}%"
    )

print("""
  Đánh giá MAPE:
    < 10%  → Dự báo rất tốt  ✅
    10-20% → Chấp nhận được  ⚠️
    > 20%  → Cần cải thiện   ❌

  Bước tiếp theo nếu cần cải thiện:
    → Thử SARIMA với seasonal period = 52
    → Tăng range grid search: p, q lên [0..5]
    → Kiểm tra residual: Ljung-Box test
""")



TỔNG KẾT KẾT QUẢ ARIMA

  Quốc gia             Order               AIC      BIC      MAE     RMSE     MAPE
  --------------------------------------------------------------------------------
  Vietnam              ❌ Lỗi: 'numpy.ndarray' object has no attribute 'iloc'
  United States        ❌ Lỗi: 'numpy.ndarray' object has no attribute 'iloc'
  China                ❌ Lỗi: 'numpy.ndarray' object has no attribute 'iloc'
  United Kingdom       ❌ Lỗi: 'numpy.ndarray' object has no attribute 'iloc'
  Brazil               ❌ Lỗi: 'numpy.ndarray' object has no attribute 'iloc'
  India                ❌ Lỗi: 'numpy.ndarray' object has no attribute 'iloc'
  South Africa         ❌ Lỗi: 'numpy.ndarray' object has no attribute 'iloc'

  Đánh giá MAPE:
    < 10%  → Dự báo rất tốt  ✅
    10-20% → Chấp nhận được  ⚠️
    > 20%  → Cần cải thiện   ❌

  Bước tiếp theo nếu cần cải thiện:
    → Thử SARIMA với seasonal period = 52
    → Tăng range grid search: p, q lên [0..5]
    → Kiểm tra residual: Ljung-Bo

17. LƯU DATASET

In [20]:
df_weekly.to_csv(OUTPUT_PATH, index=False)
print(f"  💾 Dataset đã lưu: {OUTPUT_PATH}")
print(f"  📊 Forecast plot : arima_forecast_all_countries.png")
print("=" * 65)

  💾 Dataset đã lưu: data_ready.csv
  📊 Forecast plot : arima_forecast_all_countries.png
